# Notebook 7 — Bagging, Boosting, Gradient Boosting and Stacking

## Learning objectives

- Distinguish bagging, boosting and stacking in one sentence each.
- Derive gradient boosting as steepest descent in function space.
- Train `GradientBoostingRegressor` with grid search over $T$, $\eta$, depth.
- Read the boosting curve and recognise the optimal stopping point.
- Build a stacked ensemble combining linear, RF and GBM via `StackingRegressor`.

## 7.1 The big idea: ensembles

A single estimator can only express a limited amount of structure. **Ensembles** combine multiple base learners into a more powerful predictor. We have already met one ensemble (random forests, Notebook 6). This notebook places it in a wider family.

We distinguish three ensemble strategies:

- **Bagging** (parallel, variance reduction): train $B$ estimators independently on different bootstraps; average them.
- **Boosting** (sequential, bias reduction): train $T$ estimators in order, each correcting the residual errors of the previous ones.
- **Stacking** (heterogeneous, meta-learning): train several different model families; train a *meta-model* to combine their predictions.

## 7.2 Bagging

Already covered in Notebook 6. Two further notes:

- Bagging works for any base learner, not just trees. Bagging linear regression, however, gives little benefit because linear regression is itself low-variance.
- The variance reduction is bounded below by the average pairwise correlation of base learners. Reducing that correlation (random feature subsampling, deeper trees, different random seeds) is the key extra leverage.

## 7.3 Boosting

### 7.3.1 The intuition

Train an initial weak model $f_{1}$. It makes errors. The errors (the *residuals*) from this model are then used to train a second model $f_{2}$. Combine them: $f_{1} + f_{2}$ is better than $f_{1}$ alone. Repeat. The result is a sequence

$$F_{T}(\mathbf{x}) = \sum_{t=1}^{T} \eta f_{t}(\mathbf{x}),$$

where $\eta$ is a small constant called the **learning rate** that scales each new contribution.

Boosting reduces *bias*: each new learner addresses what the previous learners got wrong. It also reduces variance for moderate $T$, but eventually overfits if $T$ is too large. Cross-validation is used to choose $T$.

### 7.3.2 Gradient boosting

The above intuition is made precise by interpreting each $f_{t}$ as a step in a gradient descent on the loss function in *function space*. Given a current ensemble $F_{t-1}$ and a per-row loss $L(F_{t-1}(\mathbf{x}_{i}), y_{i})$, define the **negative gradient** of $L$ at row $i$:

$$g_{i}^{(t)} = -\left.\frac{\partial L(F(\mathbf{x}_{i}), y_{i})}{\partial F(\mathbf{x}_{i})}\right|_{F = F_{t-1}}.$$

For squared loss this simplifies to $g_{i}^{(t)} = y_{i} - F_{t-1}(\mathbf{x}_{i})$, the ordinary residual.

The next learner $f_{t}$ is fit to *predict* $g_{i}^{(t)}$ from $\mathbf{x}_{i}$:

$$f_{t} \;\approx\; \arg\min_{f} \sum_{i=1}^{N} \bigl(g_{i}^{(t)} - f(\mathbf{x}_{i})\bigr)^{2}.$$

Then $F_{t}(\mathbf{x}) = F_{t-1}(\mathbf{x}) + \eta f_{t}(\mathbf{x})$.

In **gradient-boosted decision trees** (the `gbm` model we use), each $f_{t}$ is a *shallow* tree (typical depth 2–6). Many such shallow trees, summed, define the final model. The hyper-parameters that matter are:

- $T$ = number of boosting rounds (`n_estimators`);
- $\eta$ = learning rate (`learning_rate`);
- tree depth (`max_depth`);
- regularisation parameters (subsample fraction, L1/L2 penalties on leaf values).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (GradientBoostingRegressor, RandomForestRegressor,
                              StackingRegressor)
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_DIR = Path('data')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

csv = DATA_DIR / 'air_quality_nongzhanguan_forecast.csv'
if not csv.exists():
    csv = DATA_DIR / 'air_quality_aotizhongxin_pm25_forecast.csv'
df = pd.read_csv(csv, parse_dates=['datetime']).sort_values('datetime').reset_index(drop=True)

lag_cols = [c for c in df.columns if '_lag_' in c]
CYCLIC = ['hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'day_of_week_sin', 'day_of_week_cos']
METEO  = ['TEMP', 'PRES', 'DEWP', 'RAIN', 'WSPM']
NUMERIC = ['PM10', 'SO2', 'NO2', 'CO', 'O3'] + METEO + CYCLIC + lag_cols
TARGET = 'target_pm25_h1'
FEATURES = NUMERIC + ['wd']

cut = int(0.8 * len(df))
train_df, test_df = df.iloc[:cut], df.iloc[cut:]
print(f'Train rows: {len(train_df):,}  Test rows: {len(test_df):,}')

### The boosting curve

Track test MAE as a function of `n_estimators` to see the boosting profile. The curve should fall steeply for the first 50–100 rounds, plateau, and then either stay flat or rise (overfitting).

In [ ]:
pre_scaled = ColumnTransformer([
    ('num', StandardScaler(), NUMERIC),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['wd']),
])
x_train = pre_scaled.fit_transform(train_df[FEATURES])
x_test  = pre_scaled.transform(test_df[FEATURES])

gbm = GradientBoostingRegressor(n_estimators=300, max_depth=3, learning_rate=0.1, random_state=0)
gbm.fit(x_train, train_df[TARGET])

staged_mae = []
for i, pred in enumerate(gbm.staged_predict(x_test), start=1):
    staged_mae.append((i, mean_absolute_error(test_df[TARGET], pred)))
stg = pd.DataFrame(staged_mae, columns=['n_estimators', 'test_MAE']).set_index('n_estimators')
stg.plot(figsize=(9, 4), legend=False)
plt.title('Boosting curve: test MAE vs n_estimators')
plt.ylabel('MAE'); plt.tight_layout(); plt.show()
print(f'Lowest staged test MAE: {stg.values.min():.3f} at n_estimators = {int(stg.idxmin().iloc[0])}')

### 7.3.3 Best CV configuration on this dataset

A typical grid search exposes the three primary knobs:

```python
GradientBoostingRegressor(random_state=0), {
    'model__n_estimators': [100, 200],
    'model__max_depth': [2, 3],
    'model__learning_rate': [0.05, 0.1],
}
```

For $PM_{2.5}$ at *Nongzhanguan*, $h = 1$, `baseline_plus_lags`, an offline reference CV run picked:

```
Best CV params: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200}
Best CV score : -14.4717   (negative MAE, in PM2.5 units)
```

Producing test-set MAE 12.63 and $R^{2} = 0.939$ — essentially tied with linear regression on this metric, slightly better on MAPE. **Interpretation**: the autoregressive lag-1 dominates so completely that non-linearity adds almost nothing for $h=1$ on $PM_{2.5}$. For $O_3$ at the same horizon, GBM beats linear by $-26\%$ MAE because photochemical $O_3$ production is genuinely non-linear.

In [ ]:
grid = GridSearchCV(
    estimator=Pipeline([('pre', pre_scaled), ('gbm', GradientBoostingRegressor(random_state=0))]),
    param_grid={
        'gbm__n_estimators': [100, 200],
        'gbm__max_depth':    [2, 3],
        'gbm__learning_rate':[0.05, 0.1],
    },
    cv=TimeSeriesSplit(n_splits=3),
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
)
grid.fit(train_df[FEATURES], train_df[TARGET])
print('Best params:', grid.best_params_)
print(f'Best CV MAE: {-grid.best_score_:.3f}')
pred_gbm = grid.predict(test_df[FEATURES])
print(f'Test MAE  : {mean_absolute_error(test_df[TARGET], pred_gbm):.2f}')
print(f'Test R²   : {r2_score(test_df[TARGET], pred_gbm):.3f}')

### Head-to-head: linear, RF, GBM

In [ ]:
lr = Pipeline([('pre', pre_scaled), ('lr', LinearRegression())])
lr.fit(train_df[FEATURES], train_df[TARGET])

rf = Pipeline([('pre', pre_scaled),
               ('rf', RandomForestRegressor(n_estimators=200, max_depth=20,
                                              min_samples_leaf=5, random_state=0, n_jobs=-1))])
rf.fit(train_df[FEATURES], train_df[TARGET])

results = []
for name, model in [('LinearRegression', lr), ('RandomForest', rf), ('GBM (best)', grid.best_estimator_)]:
    pred = model.predict(test_df[FEATURES])
    results.append({'model': name,
                    'MAE': mean_absolute_error(test_df[TARGET], pred),
                    'R2':  r2_score(test_df[TARGET], pred)})
pd.DataFrame(results).set_index('model')

## 7.4 XGBoost, LightGBM, CatBoost

The vanilla `GradientBoostingRegressor` of `scikit-learn` is correct but slow on large data. Three industrial-strength gradient-boosting libraries dominate practice:

- **XGBoost** (Chen and Guestrin, 2016): regularised boosting with second-order (Newton) updates, sparse-aware split finding, parallel histogram construction. The default for tabular ML competitions.
- **LightGBM** (Ke et al., 2017): leaf-wise (rather than level-wise) tree growth, gradient-based one-side sampling, exclusive feature bundling. Faster than XGBoost on very large data.
- **CatBoost** (Prokhorenkova et al., 2018): symmetric trees, ordered boosting (eliminates target leakage), strong default for categorical features without manual encoding.

All three use the same gradient-boosting recipe; the differences are in tree construction, regularisation, and engineering. Empirically, on the Beijing dataset all three match each other within $\pm 5\%$ on every metric. All four (including the sklearn implementation) follow the `scikit-learn` estimator API:

```python
from xgboost import XGBRegressor   # pip install xgboost
XGBRegressor(
    n_estimators=400, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=0
)
```

See Exercise 7.3 below for a drop-in replacement experiment.

## 7.5 Stacking

A **stacked ensemble** combines diverse base learners through a *meta-learner* trained on their predictions. The recipe:

1. Split the training data into $K$ folds.
2. For each base learner (linear, KNN, random forest, GBM, ...) and each fold, train on the other $K-1$ folds and predict the held-out fold. Concatenate to obtain *out-of-fold* predictions for every training row.
3. Train a meta-learner (often a simple linear or logistic regression) on the matrix of out-of-fold predictions, using the original target.
4. At prediction time, each base learner is re-trained on the *full* training set; the meta-learner combines their predictions on the test data.

Stacking can squeeze 1-3 % additional accuracy from a problem when the base learners have *uncorrelated* errors. On highly autoregressive PM2.5 at $h=1$, the gain may be small because all three models lean heavily on the same `PM2.5_lag_1` feature.

> **Time-series caveat.** sklearn's `StackingRegressor` calls `cross_val_predict` internally, which requires every row to appear in exactly one test fold — i.e. the CV must produce a *partition*. `TimeSeriesSplit` deliberately does NOT form a partition: the earliest rows are never held out for testing, because no chronologically-earlier data exists to train on. Trying to use `StackingRegressor(cv=TimeSeriesSplit(...))` therefore raises `ValueError: cross_val_predict only works for partitions`. The standard workaround is to implement stacking manually with a single chronological inner-holdout split — fit the base learners on the first chunk of training data, generate honest out-of-time predictions on the second chunk, train the meta-learner on those, then refit the base learners on the full training set for the final inference. That is what the cell below does.

In [ ]:
# Time-aware manual stacking — replaces StackingRegressor for time-series data.
#
# Step 1: split TRAINING data chronologically into an inner-train block (first 75%)
#         and an inner-holdout block (next 25%).
# Step 2: fit each base learner on inner-train; predict on inner-holdout.
# Step 3: train the meta-learner on those out-of-time base-learner predictions.
# Step 4: refit every base learner on the FULL training set (so the deployed
#         stack uses all the data it has).
# Step 5: stack the refitted base-learner predictions on the chronological test
#         set and apply the meta-learner.
from sklearn.base import clone

base_learners = {
    'lr':  LinearRegression(),
    'rf':  RandomForestRegressor(n_estimators=100, max_depth=20, min_samples_leaf=5,
                                  random_state=0, n_jobs=-1),
    'gbm': GradientBoostingRegressor(n_estimators=200, max_depth=3,
                                       learning_rate=0.1, random_state=0),
}

# --- Step 1 ---
inner_cut     = int(0.75 * len(train_df))
inner_train   = train_df.iloc[:inner_cut]
inner_holdout = train_df.iloc[inner_cut:]

# Build two preprocessors: one fit on inner_train (for meta-feature generation)
# and one fit on the full training set (for the final base learners).
pre_inner = ColumnTransformer([
    ('num', StandardScaler(), NUMERIC),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['wd']),
])
pre_full  = ColumnTransformer([
    ('num', StandardScaler(), NUMERIC),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['wd']),
])
x_inner_train = pre_inner.fit_transform(inner_train[FEATURES])
x_inner_hold  = pre_inner.transform(inner_holdout[FEATURES])
x_full_train  = pre_full.fit_transform(train_df[FEATURES])
x_test_stack  = pre_full.transform(test_df[FEATURES])

# --- Step 2: meta-features = base predictions on inner-holdout ---
meta_features_train = np.zeros((len(inner_holdout), len(base_learners)))
for i, (name, est) in enumerate(base_learners.items()):
    e = clone(est)
    e.fit(x_inner_train, inner_train[TARGET])
    meta_features_train[:, i] = e.predict(x_inner_hold)

# --- Step 3: train meta-learner on out-of-time base predictions ---
meta_learner = Ridge(alpha=1.0)
meta_learner.fit(meta_features_train, inner_holdout[TARGET])
print('Meta-learner coefficients (lr, rf, gbm):', meta_learner.coef_.round(3))
print(f'Meta-learner intercept                  : {meta_learner.intercept_:.3f}')

# --- Step 4: refit base learners on the FULL training set ---
final_bases = {}
for name, est in base_learners.items():
    e = clone(est)
    e.fit(x_full_train, train_df[TARGET])
    final_bases[name] = e

# --- Step 5: stack test predictions and apply the meta-learner ---
meta_features_test = np.column_stack([e.predict(x_test_stack) for e in final_bases.values()])
pred_stack = meta_learner.predict(meta_features_test)

print()
print(f'Stacking MAE: {mean_absolute_error(test_df[TARGET], pred_stack):.2f}')
print(f'Stacking R² : {r2_score(test_df[TARGET], pred_stack):.3f}')

The meta-learner's three coefficients tell us how much weight Ridge gives to each base learner's predictions. If one coefficient dominates, the stack effectively reduces to that single base model — a sign the others are redundant. If all three contribute, the diversity is paying off.

## 7.6 When to use what

| Situation | Recommended ensemble |
|---|---|
| Strong autoregressive signal ($PM_{2.5}$ $h{=}1$) | linear or shallow GBM |
| Non-linear physics ($O_3$ photochemistry) | GBM (XGBoost/LightGBM/CatBoost) |
| Very high-cardinality categorical features | CatBoost |
| Need feature-importance interpretability | RandomForest or GBM |
| Need calibrated probabilities | GBM with isotonic calibration, or stacked logistic on top |
| Best raw accuracy, no interpretability constraints | Stacked GBM + linear meta-learner |

For a first deployment on a new pollutant, a sensible starting point is **`baseline_plus_lags` features + GBM with grid search**, which has been shown to be a strong default across multiple pollutants and horizons.

## 7.7 Common misconceptions

- **"Boosting and bagging are interchangeable."** They are not. Bagging trains models *independently*; boosting trains them *sequentially*, each correcting prior errors. They serve different purposes (variance vs. bias reduction).
- **"More boosting rounds always help."** No. Past the optimal $T$, training error keeps falling but test error rises — the model overfits. Use early stopping (validation-based) or cross-validation to choose $T$.
- **"The learning rate $\eta$ is a nuisance hyper-parameter."** It is one of the *two* most important hyper-parameters, alongside $T$. Smaller $\eta$ + more rounds usually generalises better, at the cost of training time. A common heuristic: try $\eta \in \{0.01, 0.05, 0.1\}$ and let CV pick $T$.
- **"XGBoost is always faster than `sklearn.GradientBoosting`."** True for $N \gtrsim 10^{5}$ on dense data. For our $N \approx 33\,000$ rows the speed difference is modest; the accuracy difference is small.

---
## Exercises

### Exercise 7.1 — try a smaller learning rate

*Hint: drop `learning_rate` to 0.01 and raise `n_estimators` to 1000. Theoretically this generalises better. Verify on test MAE and watch the boosting curve flatten more gradually.*

In [ ]:
# EXERCISE 7.1
# small_lr_gbm = GradientBoostingRegressor(n_estimators=1000, max_depth=3, learning_rate=0.01, random_state=0)
# TODO: fit, predict, compare MAE to the default grid winner.


### Exercise 7.2 — try GBM on O3

*Hint: load `air_quality_nongzhanguan_o3_forecast.csv`, target = `target_o3_h1`, and compare GBM to LR. Reference runs show GBM beating linear regression by roughly −26 % MAE on this pollutant. Reproduce the gap.*

In [ ]:
# EXERCISE 7.2
# o3 = pd.read_csv(DATA_DIR / 'air_quality_nongzhanguan_o3_forecast.csv', parse_dates=['datetime']).sort_values('datetime')
# TODO: rebuild NUMERIC for O3 (target lags become O3_lag_*), train both, compare.


### Exercise 7.3 — XGBoost drop-in

*Hint: install `xgboost` and replace `GradientBoostingRegressor` with `XGBRegressor(n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.8)`. The wrapper exposes the same sklearn API.*

In [ ]:
# EXERCISE 7.3
# from xgboost import XGBRegressor   # pip install xgboost
# xgb = XGBRegressor(n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.8, random_state=0)
# TODO: fit/predict, compare to sklearn GBM.


## 7.8 Chapter summary

- Ensembles combine many base learners; bagging (independent + averaged) reduces variance, boosting (sequential + residual-fitting) reduces bias, stacking (heterogeneous + meta-learner) exploits diversity.
- Gradient boosting is steepest descent in function space: each new learner predicts the negative gradient of the loss with respect to the current ensemble's prediction.
- Our `GradientBoostingRegressor` with grid-searched `learning_rate, max_depth, n_estimators` is competitive with linear regression at $h=1$ for autoregressive pollutants and dominant for non-linear ones ($O_3$).
- XGBoost / LightGBM / CatBoost are industrial-grade gradient-boosting libraries with different engineering trade-offs but the same underlying recipe; all are drop-in replacements via the `scikit-learn` estimator API.
- Choose ensemble strategy by problem: persistence-dominated targets favour bagging or shallow GBM; non-linear physics favours deeper GBM with careful regularisation; stacking buys a few extra percent at substantial complexity cost.

**Next:** Notebook 8 opens the black box and explains *why* a model predicts what it does.